<a href="https://colab.research.google.com/github/Manjit-yadav/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manjit-yadav/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Step 0 — Setup and load the data

*Imports, settings, find the CSV, and a quick look. Nothing is scored yet.*

In [12]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

DATA_NAME = "content_refresh_anonymized.csv"
OUTPUT_DIR = "work/outputs"                       # created automatically below
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "baseline_action_score.csv")


def find_data_file():
    """Look in the usual places; in Colab, ask for an upload as a last resort."""
    candidates = [
        f"/content/{DATA_NAME}",
        DATA_NAME,
        f"data/{DATA_NAME}",
        f"work/data/{DATA_NAME}",
        f"../data/{DATA_NAME}",
        f"../../data/{DATA_NAME}",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    try:
        from google.colab import files  # only exists inside Colab
        print(f"{DATA_NAME} not found - please choose the file to upload...")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".csv"):
                return name
    except ImportError:
        pass
    raise FileNotFoundError(
        f"Could not find {DATA_NAME}. Put it next to this notebook "
        f"(or in /content on Colab) and run this cell again."
    )


DATA_PATH = find_data_file()
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Rows:", len(df), "| Columns:", df.shape[1])

# The rule needs these columns - fail early with a clear message if one is missing.
REQUIRED = ["content_id", "client_id", "content_age_days", "avg_position", "ctr", "impressions_90d"]
missing_cols = [c for c in REQUIRED if c not in df.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"

print("\nMissing values in the columns the rule uses:")
display(df[REQUIRED].isna().sum().to_frame("n_missing"))
print("Duplicate content_id rows:", int(df["content_id"].duplicated().sum()))
df.head()

Loaded: /content/content_refresh_anonymized.csv
Rows: 30000 | Columns: 44

Missing values in the columns the rule uses:


,n_missing
content_id,0
client_id,0
content_age_days,0
avg_position,0
ctr,0
impressions_90d,0


Duplicate content_id rows: 0


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

**The rule in plain words.** A page goes to the human review queue when it looks *old* and/or *under-performing for where it ranks*:

1. **Age points** (from `content_age_days`): 365+ days = **2** points (stale), 180–364 days = **1** point (aging), under 180 = **0**.
2. **CTR points** (from `ctr` vs. the *median CTR of pages at a similar position*): CTR below that median = **2** points, otherwise **0**.
   - Position groups: 1–3, 4–5, 6–10, 11–20, 20+.
   - The CTR signal only counts if the page has **at least 200 impressions in 90 days**. With fewer, one click moves CTR a lot, so the number is noise.
3. **Score** = age points + CTR points (0 to 4).
4. **Action:** score 2 or more = `REVIEW_REFRESH`; score 1 = `WATCH`; score 0 = `NO_ACTION`.
5. **Ranking:** higher score first. Ties are broken by **more impressions first** (bigger possible upside), then older age first.
6. **Confidence** (a note, not part of the score): `high` = 1,000+ impressions, `medium` = 200–999, `low` = under 200 or age missing.

**Inputs used:** `content_age_days`, `ctr`, `avg_position`, `impressions_90d` — all observed over the trailing window. Nothing from the future and no product flags.

**Reason codes the rule can output:**

| Reason code | Meaning |
|---|---|
| `STALE_AND_LOW_CTR` | 365+ days old **and** CTR below the position median |
| `AGING_AND_LOW_CTR` | 180–364 days old **and** CTR below the position median |
| `LOW_CTR_FOR_POSITION` | CTR below the position median, page is under 180 days old |
| `STALE_ONLY` | 365+ days old, CTR is fine (or too little data to judge) |
| `AGING_ONLY` | 180–364 days old, CTR is fine (or too little data to judge) |
| `NO_STRONG_SIGNAL` | none of the above |

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Rule settings (change them here, then re-run from this cell downward) ----
STALE_DAYS = 365            # 2 age points at or above this
AGING_DAYS = 180            # 1 age point at or above this
MIN_IMPRESSIONS = 200       # CTR signal ignored below this (too noisy)
HIGH_CONF_IMPRESSIONS = 1000
REVIEW_SCORE = 2            # score at or above this -> REVIEW_REFRESH
POSITION_BINS = [0, 3, 5, 10, 20, np.inf]
POSITION_LABELS = ["1-3", "4-5", "6-10", "11-20", "20+"]

# ---- Quick sanity checks before locking the rule ----
check = df.copy()
check["age_bucket"] = pd.cut(
    check["content_age_days"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"],
)
check["position_bucket"] = pd.cut(
    check["avg_position"], bins=POSITION_BINS, labels=POSITION_LABELS, include_lowest=True
)

print("How many pages are in each age group?")
display(
    check.groupby("age_bucket", observed=True)
    .agg(n=("content_id", "count"), median_ctr=("ctr", "median"), median_impressions=("impressions_90d", "median"))
    .round(2)
)

print("\nMedian CTR by position group (this is the benchmark the rule uses):")
display(
    check.groupby("position_bucket", observed=True)
    .agg(n=("content_id", "count"), median_ctr=("ctr", "median"), median_impressions=("impressions_90d", "median"))
    .round(2)
)

reason_table = pd.DataFrame(
    {
        "reason_code": [
            "STALE_AND_LOW_CTR", "AGING_AND_LOW_CTR", "LOW_CTR_FOR_POSITION",
            "STALE_ONLY", "AGING_ONLY", "NO_STRONG_SIGNAL",
        ],
        "score": ["4", "3", "2", "2", "1", "0"],
        "action": ["REVIEW_REFRESH", "REVIEW_REFRESH", "REVIEW_REFRESH", "REVIEW_REFRESH", "WATCH", "NO_ACTION"],
    }
)
print("\nReason codes and the score/action each one gets:")
display(reason_table)

How many pages are in each age group?


,n,median_ctr,median_impressions
age_bucket,,,
0-90,492,0.08,294.0
91-180,11780,0.08,741.5
181-365,11368,0.06,640.5
365+,6360,0.07,842.5



Median CTR by position group (this is the benchmark the rule uses):


,n,median_ctr,median_impressions
position_bucket,,,
1-3,2346,0.00,3.0
4-5,2782,0.23,1647.0
6-10,9060,0.14,1095.0
11-20,7273,0.10,870.0
20+,8539,0.00,648.0



Reason codes and the score/action each one gets:


,reason_code,score,action
0,STALE_AND_LOW_CTR,4,REVIEW_REFRESH
1,AGING_AND_LOW_CTR,3,REVIEW_REFRESH
2,LOW_CTR_FOR_POSITION,2,REVIEW_REFRESH
3,STALE_ONLY,2,REVIEW_REFRESH
4,AGING_ONLY,1,WATCH
5,NO_STRONG_SIGNAL,0,NO_ACTION


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
work = df.copy()

# 1) Position group for each page
work["position_bucket"] = pd.cut(
    work["avg_position"], bins=POSITION_BINS, labels=POSITION_LABELS, include_lowest=True
)

# 2) Benchmark: median CTR per position group, using only pages with enough impressions
enough_data = work["impressions_90d"].fillna(0) >= MIN_IMPRESSIONS
bucket_median = work[enough_data].groupby("position_bucket", observed=True)["ctr"].median()
median_lookup = {str(k): float(v) for k, v in bucket_median.items()}
work["position_ctr_median"] = work["position_bucket"].astype(str).map(median_lookup)   # NaN if no position

# 3) CTR signal: below the benchmark AND enough impressions to trust it
work["ctr_below_position_median"] = (
    enough_data & work["ctr"].notna() & work["position_ctr_median"].notna()
    & (work["ctr"] < work["position_ctr_median"])
)

# 4) Points  (missing age -> 0 age points)
age = work["content_age_days"]
work["age_points"] = np.select([age >= STALE_DAYS, age >= AGING_DAYS], [2, 1], default=0)
work["ctr_points"] = np.where(work["ctr_below_position_median"], 2, 0)
work["score"] = work["age_points"] + work["ctr_points"]

# 5) Reason code (first matching line wins)
stale = work["age_points"] == 2
aging = work["age_points"] == 1
low_ctr = work["ctr_points"] == 2
work["reason_code"] = np.select(
    [stale & low_ctr, aging & low_ctr, low_ctr, stale, aging],
    ["STALE_AND_LOW_CTR", "AGING_AND_LOW_CTR", "LOW_CTR_FOR_POSITION", "STALE_ONLY", "AGING_ONLY"],
    default="NO_STRONG_SIGNAL",
)

# 6) Action and confidence note
work["action"] = np.select(
    [work["score"] >= REVIEW_SCORE, work["score"] == 1],
    ["REVIEW_REFRESH", "WATCH"],
    default="NO_ACTION",
)
impr = work["impressions_90d"].fillna(0)
work["confidence"] = np.select(
    [work["content_age_days"].isna() | (impr < MIN_IMPRESSIONS), impr < HIGH_CONF_IMPRESSIONS],
    ["low", "medium"],
    default="high",
)

# 7) Rank: score, then impressions (upside), then age; content_id keeps the order stable
queue = (
    work.assign(impressions_sort=impr, age_sort=work["content_age_days"].fillna(-1))
    .sort_values(
        ["score", "impressions_sort", "age_sort", "content_id"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .drop(columns=["impressions_sort", "age_sort"])
)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue = queue.rename_axis("source_row").reset_index()

OUTPUT_COLUMNS = [
    "rank", "content_id", "client_id", "score", "reason_code", "action", "confidence",
    "content_age_days", "avg_position", "position_bucket", "ctr", "position_ctr_median",
    "impressions_90d", "source_row",
]
baseline_queue = queue[OUTPUT_COLUMNS].copy()

# 8) Write the CSV (the folder is created if it does not exist)
os.makedirs(OUTPUT_DIR, exist_ok=True)
baseline_queue.to_csv(OUTPUT_PATH, index=False)
print("Saved:", os.path.abspath(OUTPUT_PATH))
print("Rows written:", len(baseline_queue))

print("\nPages by action:")
display(baseline_queue["action"].value_counts().to_frame("n_pages"))
print("Pages by reason code:")
display(baseline_queue["reason_code"].value_counts().to_frame("n_pages"))
print("Top 10 of the queue:")
display(baseline_queue.head(10))

Saved: /content/work/outputs/baseline_action_score.csv
Rows written: 30000

Pages by action:


,n_pages
action,
REVIEW_REFRESH,13298
NO_ACTION,8369
WATCH,8333


Pages by reason code:


,n_pages
reason_code,
NO_STRONG_SIGNAL,8369
AGING_ONLY,8333
LOW_CTR_FOR_POSITION,3645
STALE_ONLY,3555
AGING_AND_LOW_CTR,3293
STALE_AND_LOW_CTR,2805


Top 10 of the queue:


,rank,content_id,client_id,score,reason_code,action,confidence,content_age_days,avg_position,position_bucket,ctr,position_ctr_median,impressions_90d,source_row
0,1,content_5fe46e04994d,client_4e07408562,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,537,4.2,4-5,0.14,0.32,517715,6653
1,2,content_8c19996aa890,client_4e07408562,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,445,2.5,1-3,0.15,0.20,509252,26844
2,3,content_1a9e894be2e2,client_19581e27de,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,482,4.0,4-5,0.23,0.32,416180,29879
3,4,content_0e70a832cb7a,client_4e07408562,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,445,22.6,20+,0.04,0.06,173450,19499
4,5,content_9e08e86d0824,client_19581e27de,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,417,7.8,6-10,0.12,0.21,160851,21477
5,6,content_64373b8be1b4,client_4e07408562,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,445,6.4,6-10,0.18,0.21,153520,15353
6,7,content_09a81e8d67e2,client_19581e27de,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,389,3.5,4-5,0.26,0.32,152481,133
7,8,content_e12868d1f396,client_4e07408562,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,445,2.9,1-3,0.07,0.20,149712,16736
8,9,content_32ea6aa8091a,client_4e07408562,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,445,8.7,6-10,0.16,0.21,145501,11599
9,10,content_537a9582568a,client_19581e27de,4,STALE_AND_LOW_CTR,REVIEW_REFRESH,high,417,5.9,6-10,0.16,0.21,124870,20915


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*       

The table below is built from the numbers on each page, so the notes always match the data. Read it row by row: a **confidence note** says how much to trust the signal, and **what would make it wrong** is the most likely reason a human reviewer would say "no, skip this one".

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
WRONG_IF = {
    "STALE_AND_LOW_CTR": "Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stale content.",
    "AGING_AND_LOW_CTR": "Page is only moderately old; the low CTR may be caused by a weak title or by search features pushing results down, not by age.",
    "LOW_CTR_FOR_POSITION": "Page is fairly new; CTR may still be settling, or the query simply gets few clicks (answer shown on the results page).",
    "STALE_ONLY": "Age alone is a weak sign: if the page still gets normal CTR and traffic, the content may still be fine.",
    "AGING_ONLY": "Age alone is a weak sign: this is only 180+ days old and shows no CTR problem.",
    "NO_STRONG_SIGNAL": "No signal fired; any pick here would be a guess.",
}


def confidence_note(row):
    n = row["impressions_90d"]
    n_text = "unknown" if pd.isna(n) else f"{int(n):,}"
    base = {
        "high": "High - 1,000+ impressions in 90 days, so CTR is a stable number",
        "medium": "Medium - 200 to 999 impressions, CTR is usable but can move",
        "low": "Low - under 200 impressions (or age missing), treat as a weak hint",
    }[row["confidence"]]
    ctr_text = ""
    if pd.notna(row["ctr"]) and pd.notna(row["position_ctr_median"]):
        ctr_text = f"; CTR {row['ctr']:.2f} vs position-group median {row['position_ctr_median']:.2f}"
    return f"{base} ({n_text} impressions{ctr_text})"


top20 = baseline_queue.head(20).copy()
top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20["reason_code"].map(WRONG_IF)

review_cols = [
    "rank", "content_id", "action", "reason_code", "score", "content_age_days",
    "avg_position", "ctr", "impressions_90d", "confidence_note", "what_would_make_it_wrong",
]
display(top20[review_cols])

print("Top-20 summary")
print("  Reason codes :", top20["reason_code"].value_counts().to_dict())
print("  Confidence   :", top20["confidence"].value_counts().to_dict())
print("  Clients      :", top20["client_id"].nunique(), "distinct client(s) in the top 20")

,rank,content_id,action,reason_code,score,content_age_days,avg_position,ctr,impressions_90d,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,537,4.2,0.14,517715,"High - 1,000+ impressions in 90 days, so CTR is a stable number (517,715 impressions; CTR 0.14 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
1,2,content_8c19996aa890,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,445,2.5,0.15,509252,"High - 1,000+ impressions in 90 days, so CTR is a stable number (509,252 impressions; CTR 0.15 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
2,3,content_1a9e894be2e2,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,482,4.0,0.23,416180,"High - 1,000+ impressions in 90 days, so CTR is a stable number (416,180 impressions; CTR 0.23 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
3,4,content_0e70a832cb7a,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,445,22.6,0.04,173450,"High - 1,000+ impressions in 90 days, so CTR is a stable number (173,450 impressions; CTR 0.04 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
4,5,content_9e08e86d0824,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,417,7.8,0.12,160851,"High - 1,000+ impressions in 90 days, so CTR is a stable number (160,851 impressions; CTR 0.12 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
5,6,content_64373b8be1b4,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,445,6.4,0.18,153520,"High - 1,000+ impressions in 90 days, so CTR is a stable number (153,520 impressions; CTR 0.18 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
6,7,content_09a81e8d67e2,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,389,3.5,0.26,152481,"High - 1,000+ impressions in 90 days, so CTR is a stable number (152,481 impressions; CTR 0.26 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
7,8,content_e12868d1f396,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,445,2.9,0.07,149712,"High - 1,000+ impressions in 90 days, so CTR is a stable number (149,712 impressions; CTR 0.07 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
8,9,content_32ea6aa8091a,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,445,8.7,0.16,145501,"High - 1,000+ impressions in 90 days, so CTR is a stable number (145,501 impressions; CTR 0.16 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."
9,10,content_537a9582568a,REVIEW_REFRESH,STALE_AND_LOW_CTR,4,417,5.9,0.16,124870,"High - 1,000+ impressions in 90 days, so CTR is a stable number (124,870 impressions; CTR 0.16 vs position-group med...","Topic is evergreen so age does not matter, or the low CTR comes from the title/search-result layout rather than stal..."


Top-20 summary
  Reason codes : {'STALE_AND_LOW_CTR': 20}
  Confidence   : {'high': 20}
  Clients      : 3 distinct client(s) in the top 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak-pick flags** (checked on the top 20 and the top 100). A page is flagged when something in the data suggests a reviewer may reject it:

- `low_data` — fewer than 1,000 impressions in 90 days
- `already_top3` — average position is 3 or better (little room to gain)
- `trend_up` — traffic trend is already *up*, so it may be recovering on its own
- `recently_updated` — updated in the last 90 days, a refresh may already have happened

These flags are used **only to review** the picks. They are **not** part of the score, so they cannot change the ranking.

In [16]:
def add_weak_flags(frame):
    """Attach weak-pick flags using extra columns from the original data (review only, not scoring)."""
    out = frame.copy()
    extra = df.loc[out["source_row"]]
    extra_idx = out.index
    out["low_data"] = (out["impressions_90d"].fillna(0) < HIGH_CONF_IMPRESSIONS).to_numpy()
    out["already_top3"] = (out["avg_position"] <= 3).fillna(False).to_numpy()
    if "trend_direction" in extra.columns:
        out["trend_up"] = (extra["trend_direction"].astype(str).str.lower() == "up").to_numpy()
    else:
        out["trend_up"] = False
    if "days_since_last_update" in extra.columns:
        out["recently_updated"] = (extra["days_since_last_update"] < 90).fillna(False).to_numpy()
    else:
        out["recently_updated"] = False
    flag_cols = ["low_data", "already_top3", "trend_up", "recently_updated"]
    out["weak_flags"] = out[flag_cols].apply(lambda r: ", ".join([c for c in flag_cols if r[c]]), axis=1)
    out["is_weak"] = out[flag_cols].any(axis=1)
    return out


FLAG_COLS = ["low_data", "already_top3", "trend_up", "recently_updated"]

top20_flagged = add_weak_flags(baseline_queue.head(20))
top100_flagged = add_weak_flags(baseline_queue.head(100))

print("Share of picks with at least one weak flag:")
print(f"  top 20 : {top20_flagged['is_weak'].mean():.0%}")
print(f"  top 100: {top100_flagged['is_weak'].mean():.0%}")

print("\nHow often each flag fires (top 20 | top 100):")
display(
    pd.DataFrame(
        {"top_20": top20_flagged[FLAG_COLS].sum(), "top_100": top100_flagged[FLAG_COLS].sum()}
    )
)

weak_top20 = top20_flagged[top20_flagged["is_weak"]]
print(f"\nWeak picks inside the top 20: {len(weak_top20)}")
if len(weak_top20):
    display(weak_top20[["rank", "content_id", "reason_code", "impressions_90d", "avg_position", "weak_flags"]])
else:
    print("None of the top 20 triggered a weak flag.")

# Is the queue dominated by one client?
client_share = baseline_queue.head(100)["client_id"].value_counts(normalize=True)
print("\nClient concentration in the top 100 (share of picks):")
display(client_share.head(5).round(2).to_frame("share"))
if client_share.iloc[0] > 0.5:
    print("NOTE: more than half of the top 100 comes from one client - a reviewer may want a per-client cap.")

Share of picks with at least one weak flag:
  top 20 : 95%
  top 100: 92%

How often each flag fires (top 20 | top 100):


,top_20,top_100
low_data,0,0
already_top3,2,2
trend_up,3,14
recently_updated,19,88



Weak picks inside the top 20: 19


,rank,content_id,reason_code,impressions_90d,avg_position,weak_flags
1,2,content_8c19996aa890,STALE_AND_LOW_CTR,509252,2.5,"already_top3, recently_updated"
2,3,content_1a9e894be2e2,STALE_AND_LOW_CTR,416180,4.0,recently_updated
3,4,content_0e70a832cb7a,STALE_AND_LOW_CTR,173450,22.6,"trend_up, recently_updated"
4,5,content_9e08e86d0824,STALE_AND_LOW_CTR,160851,7.8,recently_updated
5,6,content_64373b8be1b4,STALE_AND_LOW_CTR,153520,6.4,"trend_up, recently_updated"
6,7,content_09a81e8d67e2,STALE_AND_LOW_CTR,152481,3.5,recently_updated
7,8,content_e12868d1f396,STALE_AND_LOW_CTR,149712,2.9,"already_top3, recently_updated"
8,9,content_32ea6aa8091a,STALE_AND_LOW_CTR,145501,8.7,"trend_up, recently_updated"
9,10,content_537a9582568a,STALE_AND_LOW_CTR,124870,5.9,recently_updated
10,11,content_a270c8a21b36,STALE_AND_LOW_CTR,122288,4.3,recently_updated



Client concentration in the top 100 (share of picks):


,share
client_id,
client_19581e27de,0.62
client_4e07408562,0.35
client_3fdba35f04,0.03


NOTE: more than half of the top 100 comes from one client - a reviewer may want a per-client cap.


In [17]:
# ---- Leakage check ----
SCORE_INPUTS = ["content_age_days", "ctr", "avg_position", "impressions_90d"]   # everything the score reads

# 1) Column names that look like labels, product flags, or future information
SUSPECT_WORDS = ["flag", "label", "target", "outcome", "future", "next_", "after", "recommend", "refreshed", "priority"]
suspect_cols = [c for c in df.columns if any(w in c.lower() for w in SUSPECT_WORDS)]
print("Columns in the file whose name looks like a flag/label/future field:", suspect_cols or "none")

# 2) The score must not read any of them
leaked_inputs = [c for c in SCORE_INPUTS if c in suspect_cols]
assert not leaked_inputs, f"Leakage: score reads suspect columns {leaked_inputs}"

# 3) The score must not read the short comparison windows or trend columns
window_cols = [c for c in df.columns if any(w in c for w in ["last_30d", "prev_30d", "trend_"])]
used_windows = [c for c in SCORE_INPUTS if c in window_cols]
assert not used_windows, f"Score reads window/trend columns: {used_windows}"

print("\nColumns the score reads:", SCORE_INPUTS)
print("Window/trend columns left out of the score:", window_cols)
print("\nLeakage check PASSED: score uses only age, CTR, position and 90-day impressions (observed values).")
print("Note: the CTR benchmark is a median over the same 90-day file - an observed, within-window figure, not future data.")

Columns in the file whose name looks like a flag/label/future field: none

Columns the score reads: ['content_age_days', 'ctr', 'avg_position', 'impressions_90d']
Window/trend columns left out of the score: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']

Leakage check PASSED: score uses only age, CTR, position and 90-day impressions (observed values).
Note: the CTR benchmark is a median over the same 90-day file - an observed, within-window figure, not future data.


In [18]:
# ---- Final automated self-check (all should print OK) ----
saved = pd.read_csv(OUTPUT_PATH)

checks = {
    "CSV exists": os.path.exists(OUTPUT_PATH),
    "One row per input row": len(saved) == len(df),
    "Ranks are 1..N with no gaps": saved["rank"].tolist() == list(range(1, len(df) + 1)),
    "Score is between 0 and 4": saved["score"].between(0, 4).all(),
    "Every row has a reason code and action": saved[["reason_code", "action"]].notna().all().all(),
    "Queue is sorted by score (high to low)": saved["score"].is_monotonic_decreasing,
    "Top 20 reviewed": len(top20) == 20,
}
for name, ok in checks.items():
    print(("OK   " if ok else "FAIL ") + name)
assert all(checks.values()), "One or more checks failed - read the FAIL lines above."
print("\nALL CHECKS PASSED")

OK   CSV exists
OK   One row per input row
OK   Ranks are 1..N with no gaps
OK   Score is between 0 and 4
OK   Every row has a reason code and action
OK   Queue is sorted by score (high to low)
OK   Top 20 reviewed

ALL CHECKS PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.